# 12 — Supervised volatility forecasts: HAR null, implied variance and HAR-X

**Design v0.2.** This revision fixes the two comparison flaws identified after the first real-data run (discussions/006–007):

1. Ridge penalties are selected with a **paired per-session one-standard-error rule** using the loss difference versus the best penalty, not the cross-session variability of the loss level.
2. Every nested comparison is fitted on the **larger model's complete rows**. Previously selected block penalties are frozen and only the newly added block receives a new penalty. Therefore each hierarchy step is a pure incremental-information test.

Backward HAR windows use a pre-registered **90% minimum return coverage** rule with variance sums rescaled by k / n_present. Future 30m/60m targets remain strict full-count targets.

The old 12_frozen_spec.json belongs to design v0.1 and is invalid for confirmation. This notebook writes and reads 12_frozen_spec_v02.json. The binding test remains the first 30 completed sessions after 2026-10-07.

In [ ]:
from pathlib import Path
import os, sys, time, json, subprocess, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)

load_dotenv()
DATA_ROOT = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve()
DB_PATH = DATA_ROOT / "database" / "research.duckdb"
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

for _m in [m for m in sys.modules if m.split(".")[0] == "data_engineering"]:
    del sys.modules[_m]

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling import full_information as fi
from data_engineering.modeling import vol_targets as vt
from data_engineering.modeling import vol_har as vh
from data_engineering.modeling.clock_baseline import CLOCK_BASELINE_VERSION
from data_engineering.modeling.hsmm.design import build_design
from data_engineering.modeling.hsmm.selection import blocked_folds

if vt.VOL_TARGETS_VERSION < "0.1" or vh.VOL_HAR_VERSION < "0.1":
    raise RuntimeError("Old Module 12 code: git pull and restart the kernel.")

DESIGN_VERSION = "0.2"
MODE = os.getenv("NB12_MODE", "development")
ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
DEV_START, SCREEN_TRAIN_END, DEV_END = "2026-09-22", "2026-09-28", "2026-10-07"
MIN_FEATURE_COVERAGE = float(os.getenv("NB12_MIN_FEATURE_COVERAGE", "0.45"))
BACK_MIN_COVERAGE = float(os.getenv("NB12_BACK_MIN_COVERAGE", "0.90"))
N_BOOT = int(os.getenv("NB12_BOOT", "2000"))
FREEZE = os.getenv("NB12_FREEZE", "1") == "1"
REFREEZE = os.getenv("NB12_REFREEZE", "0") == "1"
SPEC_PATH = PROJECT_ROOT / "notebooks" / "12_frozen_spec_v02.json"
CONFIRM_MIN_SESSIONS = 30
HORIZONS = list(vt.HORIZONS)

print(
    f"mode {MODE} | design {DESIGN_VERSION} | vol_targets {vt.VOL_TARGETS_VERSION} | "
    f"vol_har {vh.VOL_HAR_VERSION} | clock {CLOCK_BASELINE_VERSION} | "
    f"development {DEV_START} -> {DEV_END} (screening fit through {SCREEN_TRAIN_END}) | "
    f"backward min coverage {BACK_MIN_COVERAGE:.0%} | lambda grid {vh.LAMBDA_GRID}"
)

## 1. Load the state table

In development mode only sessions up to 2026-10-07 are read: confirmation data is never touched before the specification is frozen.

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])

required = {
    "state_id", "state_time", "underlying", "return_5m", "futures_index_price", "is_weekend",
    "in_settlement_window", "vwap", "volume_usd", "futures_tradable_mid_price", "trade_count",
    "perp_integrity_ok", "known_exchange_event", "integrity_flag_trades_outside_bbo",
    "microstructure_features_ready", "instability_features_ready", "option_snapshot_ready",
    "option_atm_quote_ok", "option_shape_quote_ok", "fit_ok",
    "option_iv_atm_forward",
}
optional = {"open", "high", "low", "close"}            # perp trade OHLC: range regressors only
missing_required = sorted(required - cols)
if missing_required:
    raise RuntimeError(f"time_series_feature_state lacks {missing_required}")
want = sorted((required | optional | {s.column for s in fi.FEATURE_SPECS}) & cols)
if optional - cols:
    print(f"Perp OHLC columns missing ({sorted(optional - cols)}): Parkinson / Garman-Klass regressors will be empty.")

DEV_LAST_BAR = pd.Timestamp(f"{DEV_END} 08:00", tz="UTC")     # closes the last development expiry
dev_filter = "" if MODE == "confirmation" else (
    f"WHERE state_time <= TIMESTAMPTZ '{DEV_LAST_BAR.isoformat()}'"
)
df = con.execute(
    f"SELECT {', '.join(want)} FROM time_series_feature_state {dev_filter} ORDER BY underlying, state_time"
).df()
con.close()

df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
df = df[df["underlying"].isin(ASSETS)].reset_index(drop=True)
df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["session_date"] = ri.session_key(df["state_time"])
if not (df["session_date"].to_numpy() == vt.expiry_time(df["state_time"]).sub(pd.Timedelta(hours=8)).dt.strftime("%Y-%m-%d").to_numpy()).all():
    raise RuntimeError("Session keys of regime_inputs and vol_targets disagree.")

if MODE != "confirmation":
    # The bar closing at DEV_END 08:00 carries the last return of the final development
    # expiry; it is kept only as a target increment (its session is never an origin).
    keep = ((df["session_date"] >= DEV_START) & (df["session_date"] <= DEV_END)) | (df["state_time"] == DEV_LAST_BAR)
    df = df[keep].reset_index(drop=True)
    assert df["state_time"].max() <= DEV_LAST_BAR, "confirmation data leaked into development"

DEV_SESSIONS = sorted(s for s in df["session_date"].unique() if DEV_START <= s <= DEV_END)
SCREEN_TRAIN = {s for s in DEV_SESSIONS if s <= SCREEN_TRAIN_END}
SCREEN_EVAL = {s for s in DEV_SESSIONS if s > SCREEN_TRAIN_END}
print(
    f"Rows {len(df):,} | development sessions {len(DEV_SESSIONS)} "
    f"(screening fit {len(SCREEN_TRAIN)}, screening evaluation {len(SCREEN_EVAL)})"
)
if len(SCREEN_TRAIN) < 5 or len(SCREEN_EVAL) < 3:
    raise RuntimeError("Module 12 needs the Sep 22-28 sessions and the later development sessions.")

## 2. Inputs per asset

* **Future targets stay strict.** Every forward return in the 30m/60m target must be present and contiguous.
* **Backward HAR windows use at least 90% observed returns**, rescaled by k / n_present. With the current windows this still requires 6/6 returns at 30m, at least 11/12 at 1h, and at least 44/48 at 4h.
* D features remain origin-time causal.
* Feature eligibility is decided on training rows only.
* Every nested comparison is fitted on identical rows within the pair.

In [ ]:
def asset_frame(asset, frame=None):
    frame = df if frame is None else frame
    return frame[frame["underlying"] == asset].sort_values("state_time").reset_index(drop=True)


def backward_sum_coverage(x, times, k, min_fraction=BACK_MIN_COVERAGE):
    x = np.asarray(x, float)
    n = len(x)
    out = np.full(n, np.nan)
    if n < k:
        return out

    j = np.arange(k - 1, n)
    i = j - (k - 1)
    t = pd.to_datetime(times, utc=True).to_numpy()
    contiguous = (t[j] - t[i]) == (j - i) * vt.BAR.to_timedelta64()

    finite = np.isfinite(x)
    csum = np.r_[0.0, np.cumsum(np.where(finite, x, 0.0))]
    ccnt = np.r_[0, np.cumsum(finite.astype(int))]
    s = csum[j + 1] - csum[i]
    cnt = ccnt[j + 1] - ccnt[i]

    need = int(np.ceil(min_fraction * k))
    valid = contiguous & (cnt >= need) & (cnt > 0)
    scaled = np.where(cnt > 0, s * (k / np.maximum(cnt, 1)), np.nan)
    out[j] = np.where(valid, scaled, np.nan)
    return out


def build_panel_v02(g):
    panel = vt.build_panel(g)
    r2 = panel.r ** 2
    panel.back_rv = {
        w: backward_sum_coverage(r2, panel.times, k, BACK_MIN_COVERAGE)
        for w, k in vt.BACK_WINDOWS.items()
    }
    return panel


def d_features(g, nuis):
    cbs = nuis.clock_set()
    clock_frame = g[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]]
    f = cbs.transform(clock_frame)["return_5m__clock_vol_factor"].fillna(1.0).to_numpy()
    d = build_design(g, f)
    D = np.full(len(g), np.nan)
    src = d.frame["source_row"].to_numpy(int)
    D[src - 1] = d.frame["D"].to_numpy(float)
    t = pd.to_datetime(g["state_time"], utc=True)
    contig = np.r_[False, (t.diff().iloc[1:] == vt.BAR).to_numpy()]
    absD = np.abs(D)
    dabs = np.where(contig, absD - np.r_[np.nan, absD[:-1]], np.nan)
    return pd.DataFrame({"absD": absD, "D": D, "d_absD": dabs})


def raw_features(g, nuis):
    raw, used, _ = fi.raw_feature_frame(g)
    return pd.concat([raw.reset_index(drop=True), d_features(g, nuis)], axis=1), used


def eligibility(mat, used_specs, train_rows, h):
    X = mat.X[h]
    cov = X.loc[train_rows].notna().mean()
    nun = X.loc[train_rows].nunique(dropna=True)
    avail = {c for c in X.columns if cov.get(c, 0.0) >= MIN_FEATURE_COVERAGE and nun.get(c, 0) >= 3}
    specs_ok = [s for s in used_specs if s.name in avail]
    return avail, fi.block_map(specs_ok), cov


def prepare(asset, train_sessions, frame=None):
    g = asset_frame(asset, frame)
    panel = build_panel_v02(g)
    nuis = vh.fit_nuisance(g, panel, train_sessions)
    raw, used = raw_features(g, nuis)
    mat = vh.materialize(panel, nuis, raw)
    return dict(g=g, panel=panel, nuis=nuis, raw=raw, used=used, mat=mat)


INPUTS = {a: prepare(a, SCREEN_TRAIN) for a in ASSETS}

for a, b in INPUTS.items():
    vt.alignment_selftest(b["g"])
print("Alignment self-test passed for", ", ".join(INPUTS))

cov_rows = []
for a, b in INPUTS.items():
    m, P = b["mat"], b["panel"]
    for h in HORIZONS:
        dev = np.isin(P.session_date, DEV_SESSIONS)
        h_ok = np.isfinite(m.H[h][vh.H_NAMES].to_numpy(float)).all(axis=1)
        cov_rows.append({
            "underlying": a, "horizon": h, "origins": int(dev.sum()),
            "target_share": float(np.isfinite(m.y[h][dev]).mean()),
            "HAR_share": float(h_ok[dev].mean()),
            "W_imp_share": float(np.isfinite(m.X[h]["log_W_imp_alloc"].to_numpy()[dev]).mean()),
            "D_share": float(np.isfinite(b["raw"]["D"].to_numpy()[dev]).mean()),
            "log10_eps": np.log10(b["nuis"].eps_target[h]),
            "iv_divisor": P.iv_divisor,
            "median_iv": float(np.nanmedian(P.iv)),
        })
print("Target and input coverage (development sessions):")
display(pd.DataFrame(cov_rows).round(4))

## 3. Target diagnostics

Overlap makes consecutive 30m / 60m targets strongly autocorrelated; inference is session-blocked throughout. The terminal target is compared with its log-return analogue (004 §3) and with the clock-allocated implied variance of the settlement average.

In [ ]:
diag = []
for a, b in INPUTS.items():
    m, P = b["mat"], b["panel"]
    for h in HORIZONS:
        y = pd.Series(m.y[h])
        lag = vt.HORIZONS[h]
        diag.append({
            "underlying": a, "horizon": h, "n": int(y.notna().sum()),
            "mean": y.mean(), "sd": y.std(), "skew": y.skew(),
            "ac_lag1": y.autocorr(1), f"ac_lag_m": y.autocorr(lag),
            "corr_y_l30m": pd.Series(m.y[h]).corr(m.H[h]["l30m"]),
        })
print("Log-RV targets:")
display(pd.DataFrame(diag).round(3))

term = []
for a, b in INPUTS.items():
    s = b["mat"].settle
    ok = np.isfinite(s.rv_settle) & (s.rv_settle > 0)
    w = ok & np.isfinite(s.W_imp_settle) & (s.W_imp_settle > 0)
    term.append({
        "underlying": a,
        "finite_share": float(ok.mean()),
        "median_level_over_log": float(np.nanmedian(s.rv_settle[ok] / s.rv_settle_log[ok])),
        "p99_abs_log_diff": float(np.nanquantile(np.abs(np.log(s.rv_settle[ok] / s.rv_settle_log[ok])), 0.99)),
        "corr_log_rv_settle_log_W": float(np.corrcoef(np.log(s.rv_settle[w]), np.log(s.W_imp_settle[w]))[0, 1]) if w.sum() > 10 else np.nan,
        "median_log_rv_over_W": float(np.nanmedian(np.log(s.rv_settle[w] / s.W_imp_settle[w]))) if w.any() else np.nan,
    })
print("Provisional terminal target (descriptive only):")
display(pd.DataFrame(term).round(4))

## 4. Development screening

Fit on 2026-09-22 → 09-28, evaluate on 09-29 → 10-07. Inside the fit window, $\lambda$ is chosen by blocked expanding-window CV on QLIKE with every nuisance quantity refitted per fold, using a **one-standard-error rule toward the larger penalty** (004 §7). M1c is recalibrated on out-of-fold M1 forecasts. OLS on M4 is a diagnostic; Elastic Net on M4 is the sparse challenger.

These nine evaluation sessions were inspected in Modules 08–11: the readout below is screening, not a test.

In [ ]:
def fit_window(asset, b, train_sessions, h):
    """CV, penalty choice and final fits for one asset / horizon / window."""
    g, P, mat = b["g"], b["panel"], b["mat"]
    tr = np.isin(mat.sessions, sorted(train_sessions))
    avail, blocks, cov = eligibility(mat, b["used"], tr, h)
    specs = vh.model_specs(blocks, avail)
    folds = blocked_folds(sorted(train_sessions), n_folds=3, min_train_share=0.45)

    def build(trs):
        return vh.materialize(P, vh.fit_nuisance(g, P, trs), b["raw"])

    cv, extra = vh.cross_validate(build, folds, h, specs, en_grid=EN_GRID)
    lam = {}
    for m in ("M2a", "M2b", "M3", "M4"):
        lam[m] = vh.one_se_choice(cv, m)[0] if specs[m] else 0.0
        if not np.isfinite(lam[m]):
            lam[m] = max(vh.LAMBDA_GRID)
    en_alpha = vh.one_se_choice(cv, "M4", "elasticnet")[0] if EN_GRID else np.nan

    fits = {"M0": vh.fit_vol(mat, h, "M0", [], tr)}
    fits["M1"] = vh.fit_vol(mat, h, "M1", [], tr)
    fits["M1c"] = vh.calibrate_m1(fits["M1"], extra["oof_m1"])
    for m in ("M2a", "M2b", "M3", "M4"):
        fits[m] = vh.fit_vol(mat, h, m, specs[m], tr, lam=lam[m])
    fits["M4_ols"] = vh.fit_vol(mat, h, "M4_ols", specs["M4"], tr, lam=0.0)
    if EN_GRID and np.isfinite(en_alpha):
        fits["M4_en"] = vh.fit_vol(mat, h, "M4_en", specs["M4"], tr, kind="elasticnet", lam=en_alpha)
    return dict(specs=specs, avail=avail, blocks=blocks, cv=cv, lam=lam, en_alpha=en_alpha, fits=fits, coverage=cov)


SCREEN = {}
t_all = time.perf_counter()
for a, b in INPUTS.items():
    t0 = time.perf_counter()
    SCREEN[a] = {h: fit_window(a, b, SCREEN_TRAIN, h) for h in HORIZONS}
    print(f"{a}: screening fits in {time.perf_counter() - t0:.0f}s")
print(f"total {time.perf_counter() - t_all:.0f}s")

lam_rows = []
for a, by_h in SCREEN.items():
    for h, r in by_h.items():
        lam_rows.append({"underlying": a, "horizon": h, **{f"lam_{m}": v for m, v in r["lam"].items()},
                         "en_alpha_M4": r["en_alpha"], **{f"n_{m}": len(r["specs"][m]) for m in ("M2b", "M3", "M4")}})
print("Selected penalties (one-SE toward larger) and model sizes:")
display(pd.DataFrame(lam_rows))

In [ ]:
SCORES, metrics = {}, []
for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))
    SCORES[a] = {}
    for h, r in by_h.items():
        SCORES[a][h] = {m: vh.row_scores(fv, mat, h) for m, fv in r["fits"].items()}
        y = mat.y[h]
        dy = y - mat.delta_base[h]
        ref = {k: SCORES[a][h][k]["yhat"].to_numpy() for k in ("M1", "M2b") if k in SCORES[a][h]}
        for m, sc in SCORES[a][h].items():
            yhat = sc["yhat"].to_numpy()
            rows = te & np.isfinite(sc["qlike"].to_numpy())
            rec = {
                "underlying": a, "horizon": h, "model": m, "n": int(rows.sum()),
                "qlike": float(sc["qlike"][rows].mean()), "log_rmse": float(np.sqrt(sc["sqerr"][rows].mean())),
                "ls_gauss": float(sc["ls_gauss"][rows].mean()), "ls_t": float(sc["ls_t"][rows].mean()),
                "r2": vh.r2(y[rows], yhat[rows]), "r2_change_form": vh.r2(dy[rows], (yhat - mat.delta_base[h])[rows]),
            }
            for k, yr in ref.items():
                both = rows & np.isfinite(yr)
                rec[f"dR2_vs_{k}"] = vh.r2(y[both], yhat[both]) - vh.r2(y[both], yr[both])
            rec.update(vh.mz(y[rows], yhat[rows], mat.sessions[rows], n_boot=500))
            metrics.append(rec)
metrics = pd.DataFrame(metrics)
print("Screening metrics on 2026-09-29 -> 10-07 (each model on its own complete rows):")
display(metrics.round(4))

### Paired comparisons

Positive = the first model is better. `qlike_*` and `sqerr_*` are session sums of loss reductions; `ls_t_*` of Student-t log-score gains. Each pair is scored on the rows both models can score. The non-overlapping view keeps only session-aligned origins every $h$.

In [ ]:
PAIRS = vh.HIERARCHY + [("M1c", "M1"), ("M4_en", "M4"), ("M4_ols", "M4")]
cmp_rows = []
for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))
    for h in by_h:
        nov = vh.non_overlapping(mat.tte_min, h)
        for x, y_ in PAIRS:
            if x not in SCORES[a][h] or y_ not in SCORES[a][h]:
                continue
            for view, rows in (("all", te), ("non_overlapping", te & nov)):
                c = vh.compare(SCORES[a][h][x], SCORES[a][h][y_], mat.sessions, rows, n_boot=N_BOOT)
                cmp_rows.append({"underlying": a, "horizon": h, "model": x, "vs": y_, "view": view, **c})
comparisons = pd.DataFrame(cmp_rows)
show = ["underlying", "horizon", "model", "vs", "view", "n_rows", "qlike_n_sessions", "qlike_mean_session_diff",
        "qlike_boot_lo", "qlike_boot_hi", "qlike_sessions_won", "sqerr_mean_session_diff", "ls_t_mean_session_diff"]
display(comparisons[show].round(4))

### Population retained by each model (004 §11)

Option-based models score only quote-backed rows outside the settlement window. This table shows what population remains.

In [ ]:
def tte_bucket(tte):
    return pd.cut(tte / 60.0, [-1, 4, 8, 16, 24.5], labels=["0-4h", "4-8h", "8-16h", "16-24h"])

sel = []
for a, by_h in SCREEN.items():
    b = INPUTS[a]
    mat, raw = b["mat"], b["raw"]
    dev = np.isin(mat.sessions, DEV_SESSIONS)
    spread = raw["log_option_spread_med"].to_numpy() if "log_option_spread_med" in raw else np.full(len(raw), np.nan)
    for h, r in by_h.items():
        elig = dev & np.isfinite(mat.y[h]) & np.isfinite(mat.H[h][vh.H_NAMES].to_numpy(float)).all(axis=1)
        for m in ("M1", "M2a", "M2b", "M3", "M4"):
            keep = elig & vh.complete(mat, h, r["specs"][m])
            rec = {"underlying": a, "horizon": h, "model": m, "eligible": int(elig.sum()), "retained": int(keep.sum()),
                   "retained_share": float(keep.sum() / max(elig.sum(), 1)),
                   "weekend_share_retained": float(INPUTS[a]["panel"].weekend[keep].mean()) if keep.any() else np.nan,
                   "min_session_share": float(pd.Series(keep[elig]).groupby(mat.sessions[elig]).mean().min()),
                   "median_y_retained": float(np.nanmedian(mat.y[h][keep])) if keep.any() else np.nan,
                   "median_y_excluded": float(np.nanmedian(mat.y[h][elig & ~keep])) if (elig & ~keep).any() else np.nan,
                   "median_log_opt_spread_retained": float(np.nanmedian(spread[keep])) if keep.any() else np.nan,
                   "median_log_opt_spread_excluded": float(np.nanmedian(spread[elig & ~keep])) if (elig & ~keep).any() else np.nan}
            buckets = pd.Series(keep[elig]).groupby(tte_bucket(mat.tte_min[elig]).astype(str)).mean()
            rec.update({f"share_{k}": v for k, v in buckets.items()})
            sel.append(rec)
selection = pd.DataFrame(sel)
display(selection.round(4))

### Coefficients

HAR + clock coefficients of M1 (persistence structure), and the largest standardized added-block coefficients of M3 / M4.

In [ ]:
har = []
top = []
for a, by_h in SCREEN.items():
    for h, r in by_h.items():
        m1 = r["fits"]["M1"]
        har.append({"underlying": a, "horizon": h, **dict(zip(m1.h_names, np.round(m1.phi, 4))), "smear": m1.smear})
        for m in ("M3", "M4"):
            fv = r["fits"][m]
            if fv.x_names:
                s = pd.Series(fv.gamma, index=fv.x_names)
                for name, v in s.reindex(s.abs().sort_values(ascending=False).index).head(10).items():
                    top.append({"underlying": a, "horizon": h, "model": m, "feature": name, "coef": v})
print("M1 HAR + clock coefficients:")
display(pd.DataFrame(har))
print("Largest added-block coefficients (standardized, after shrinkage):")
display(pd.DataFrame(top).round(4))

### Physical forecast vs clock-allocated implied variance (descriptive)

$\log(\widehat W^{real}/\widetilde W^{imp})$ against $\log(RV/\widetilde W^{imp})$ on the evaluation rows. The median realized-to-implied log ratio is the in-sample variance-risk-premium proxy. This is not a P&L statement: a gap can be a rational premium (004 §10).

In [ ]:
econ = []
for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))
    for h, r in by_h.items():
        lW = mat.X[h]["log_W_imp_alloc"].to_numpy()
        real = np.log(mat.level[h]) - lW
        for m in ("M1", "M2b", "M3"):
            sc = SCORES[a][h][m]
            gap_hat = sc["yhat"].to_numpy() + np.log(r["fits"][m].smear) - lW
            ok = te & np.isfinite(gap_hat) & np.isfinite(real)
            econ.append({"underlying": a, "horizon": h, "model": m, "n": int(ok.sum()),
                         "corr_gap_hat_gap_real": float(np.corrcoef(gap_hat[ok], real[ok])[0, 1]) if ok.sum() > 10 else np.nan,
                         "median_log_rv_over_W": float(np.median(real[ok])) if ok.any() else np.nan})
display(pd.DataFrame(econ).round(4))

## 5. Freeze the specification

All nuisance quantities, eligibility, penalties and coefficients are refitted on **all development sessions** (2026-09-22 → 10-07; CV folds inside them) and written to `notebooks/12_frozen_spec.json` with the code commit. Commit that file **before** scoring any confirmation session. An existing spec is never overwritten unless `NB12_REFREEZE=1`.

In [ ]:
def git_commit():
    try:
        return subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


if MODE == "development" and FREEZE:
    if SPEC_PATH.exists() and not REFREEZE:
        print(f"{SPEC_PATH.name} exists: not overwritten (set NB12_REFREEZE=1 to replace it before confirmation).")
    else:
        spec = {"meta": {
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "git_commit": git_commit(), "vol_targets": vt.VOL_TARGETS_VERSION, "vol_har": vh.VOL_HAR_VERSION,
            "clock_baseline": CLOCK_BASELINE_VERSION, "full_information": fi.FULL_INFORMATION_VERSION,
            "development_sessions": DEV_SESSIONS, "confirmation_starts_after": DEV_END,
            "confirmation_min_sessions": CONFIRM_MIN_SESSIONS, "min_feature_coverage": MIN_FEATURE_COVERAGE,
            "lambda_grid": list(vh.LAMBDA_GRID), "en_grid": list(EN_GRID), "horizons": vt.HORIZONS,
            "primary_score": "QLIKE (Duan smearing)", "hierarchy": [list(p) for p in vh.HIERARCHY[1:]],
        }, "assets": {}}
        t0 = time.perf_counter()
        FROZEN_INPUTS = {}
        for a in ASSETS:
            b = prepare(a, set(DEV_SESSIONS))
            FROZEN_INPUTS[a] = b
            entry = {"nuisance": b["nuis"].to_dict(), "horizons": {}}
            for h in HORIZONS:
                r = fit_window(a, b, set(DEV_SESSIONS), h)
                entry["horizons"][h] = {
                    "specs": r["specs"], "lambda": r["lam"], "en_alpha": r["en_alpha"],
                    "models": {m: fv.to_dict() for m, fv in r["fits"].items()},
                }
            spec["assets"][a] = entry
            print(f"{a}: frozen ({time.perf_counter() - t0:.0f}s)")
        SPEC_PATH.write_text(json.dumps(spec, indent=1, default=float))
        print(f"Wrote {SPEC_PATH} ({SPEC_PATH.stat().st_size / 1e6:.2f} MB). Commit it before any confirmation scoring.")

## 6. Confirmation (binding after 30 completed sessions)

Run with `NB12_MODE=confirmation`. The frozen nuisance, eligibility and coefficients are applied unchanged to every completed session after 2026-10-07. With fewer than 30 sessions the readout is a **non-binding checkpoint** and changes no model choice.

In [ ]:
if MODE == "confirmation":
    spec = json.loads(SPEC_PATH.read_text())
    print(f"Frozen spec from commit {spec['meta']['git_commit'][:10]} created {spec['meta']['created_utc']}")
    conf_sessions = sorted(
        s for s in df["session_date"].unique()
        if s > spec["meta"]["confirmation_starts_after"]
        and pd.Timestamp(f"{s} 08:00", tz="UTC") <= df["state_time"].max()
    )
    binding = len(conf_sessions) >= spec["meta"]["confirmation_min_sessions"]
    print(f"Completed confirmation sessions: {len(conf_sessions)} -> "
          f"{'BINDING readout' if binding else 'NON-BINDING checkpoint'}")

    conf_rows, conf_metrics = [], []
    for a in ASSETS:
        entry = spec["assets"][a]
        g = asset_frame(a)
        panel = vt.build_panel(g)
        nuis = vh.Nuisance(**entry["nuisance"])
        raw, _ = raw_features(g, nuis)
        mat = vh.materialize(panel, nuis, raw)
        cr = np.isin(mat.sessions, conf_sessions)
        for h, hz in entry["horizons"].items():
            sc = {m: vh.row_scores(vh.FittedVol.from_dict(d), mat, h) for m, d in hz["models"].items()}
            nov = vh.non_overlapping(mat.tte_min, h)
            for m, s in sc.items():
                rows = cr & np.isfinite(s["qlike"].to_numpy())
                conf_metrics.append({"underlying": a, "horizon": h, "model": m, "n": int(rows.sum()),
                                     "qlike": float(s["qlike"][rows].mean()),
                                     "r2": vh.r2(mat.y[h][rows], s["yhat"].to_numpy()[rows])})
            for x, y_ in vh.HIERARCHY + [("M1c", "M1")]:
                for view, rows in (("all", cr), ("non_overlapping", cr & nov)):
                    c = vh.compare(sc[x], sc[y_], mat.sessions, rows, n_boot=N_BOOT)
                    conf_rows.append({"underlying": a, "horizon": h, "model": x, "vs": y_, "view": view, **c})
    confirmation = pd.DataFrame(conf_rows)
    display(pd.DataFrame(conf_metrics).round(4))
    display(confirmation[show].round(4))
else:
    print("Development mode: confirmation not run.")

## 7. Interpretation gates

1. **Specification checks** (development): targets and inputs present for every asset; HAR coefficients positive and plausible; M1 beats M0; no model's QLIKE explodes. A failure here is fixed before freezing, not interpreted.
2. **Sensor hierarchy** (binding, confirmation only): M2a vs M1, M2b vs M2a, M3 vs M2b, M4 vs M3, judged by the session-paired QLIKE difference and its whole-session bootstrap interval, with the non-overlapping view required to agree in sign.
3. **Module 13 gate** (independent of 2): no alignment failures and finite, stable HAR forecasts. If M1 has an unconditional calibration bias, the training-only recalibrated **M1c** is the null that switching models must beat (004 §8).
4. **No economic claim.** A forecast–implied gap is a candidate only if it later predicts executable option P&L net of spread, fees and hedge costs.
5. **Terminal target** stays provisional until the settlement-average path simulation is built.